# Current clamp waveforms and HH responses

This example applies three current-clamp waveforms to the same one-CV Hodgkin-Huxley cell template in independent runs: a DC pulse, a finite sine wave, and an arbitrary biphasic waveform. Each run starts from the same initial state, so its voltage response can be compared directly.

Clamp current is recorded through `cell.clamps[...]`. These samples are the cached values consumed by the solver: for a fixed main step beginning at `t`, BrainCell evaluates the clamp once at `t + 0.5 * dt` and holds that value for the complete step. Clamp windows are left-closed and right-open.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import braincell
import brainunit as u
from braincell.filter import AllRegion, RootLocation

DT = 0.025 * u.ms
DURATION = 50.0 * u.ms

## Cell and waveform definitions

`FunctionClamp` accepts any JAX-compatible function from absolute simulation time to current. The function below combines positive and negative Gaussian pulses and explicitly gates them to a finite interval.

In [ ]:
def arbitrary_current(t):
    t_ms = t.to_decimal(u.ms)
    waveform = (
        0.18 * u.math.exp(-0.5 * ((t_ms - 14.0) / 1.5) ** 2)
        - 0.10 * u.math.exp(-0.5 * ((t_ms - 25.0) / 2.5) ** 2)
    )
    active = (t_ms >= 5.0) & (t_ms < 35.0)
    return u.math.where(active, waveform, 0.0) * u.nA


def build_hh_cell(clamp):
    soma = braincell.Branch.from_lengths(
        lengths=[20.0] * u.um,
        radii=[10.0, 10.0] * u.um,
        type="soma",
    )
    cell = braincell.Cell(
        braincell.Morphology.from_root(soma, name="soma"),
        cv_policy=braincell.CVPerBranch(),
        V_init=-65.0 * u.mV,
        solver="staggered",
    )
    cell.paint(
        AllRegion(),
        braincell.mech.CableProperty(
            resting_potential=-54.3 * u.mV,
            membrane_capacitance=1.0 * u.uF / u.cm**2,
            axial_resistivity=100.0 * u.ohm * u.cm,
        ),
        braincell.mech.Ion("SodiumFixed", E=50.0 * u.mV),
        braincell.mech.Ion("PotassiumFixed", E=-77.0 * u.mV),
        braincell.mech.Channel(
            "IL", name="leak", g_max=0.3 * u.mS / u.cm**2, E=-54.3 * u.mV
        ),
        braincell.mech.Channel(
            "Na_HH1952", name="na", g_max=120.0 * u.mS / u.cm**2
        ),
        braincell.mech.Channel(
            "K_HH1952", name="k", g_max=36.0 * u.mS / u.cm**2
        ),
    )
    cell.place(RootLocation(0.5), clamp)
    cell.soma.record("voltage", braincell.observe.state("v"))
    cell.clamps[type(clamp).__name__].record("current")
    cell.init_state()
    return cell

## Independent stimulation protocols

Clamp declarations do not have semantic names. `cell.clamps["CurrentClamp"]` and the corresponding type strings select them by clamp type. Here each independently simulated cell has exactly one selected clamp.

In [ ]:
protocols = {
    "DC pulse": braincell.mech.CurrentClamp(
        delay=5.0 * u.ms, durations=30.0 * u.ms, amplitudes=0.12 * u.nA
    ),
    "Sine wave": braincell.mech.SineClamp(
        amplitude=0.12 * u.nA,
        frequency=100.0 * u.Hz,
        offset=0.06 * u.nA,
        delay=5.0 * u.ms,
        duration=30.0 * u.ms,
    ),
    "Arbitrary biphasic": braincell.mech.FunctionClamp(fn=arbitrary_current),
}

results = {}
for label, clamp in protocols.items():
    cell = build_hh_cell(clamp)
    results[label] = cell.run(dt=DT, duration=DURATION)

## Recorded solver input

Voltage state samples are taken at main-step starts. Clamp samples carry their actual midpoint timestamps, so the two time axes differ by exactly half a step.

In [ ]:
for label, result in results.items():
    current = result.samples["current"]
    voltage = result.samples["voltage"]
    current_time_ms = np.asarray(current.time.to_decimal(u.ms))
    voltage_time_ms = np.asarray(voltage.time.to_decimal(u.ms))
    current_nA = np.asarray(current.values.to_decimal(u.nA))[:, 0]
    voltage_mV = np.asarray(voltage.values.to_decimal(u.mV))[:, 0]

    np.testing.assert_allclose(
        current_time_ms - voltage_time_ms, 0.5 * DT.to_decimal(u.ms), rtol=0.0, atol=1e-6
    )
    assert np.isfinite(current_nA).all()
    assert np.isfinite(voltage_mV).all()
    print(
        f"{label:20s}  I=[{current_nA.min(): .3f}, {current_nA.max(): .3f}] nA  "
        f"V=[{voltage_mV.min(): .1f}, {voltage_mV.max(): .1f}] mV"
    )

dc_block = results["DC pulse"].samples["current"]
dc_time_ms = np.asarray(dc_block.time.to_decimal(u.ms))
dc_current_nA = np.asarray(dc_block.values.to_decimal(u.nA))[:, 0]
active_start = int(round(5.0 / DT.to_decimal(u.ms)))
active_stop = int(round(35.0 / DT.to_decimal(u.ms)))
np.testing.assert_allclose(dc_current_nA[:active_start], 0.0)
np.testing.assert_allclose(dc_current_nA[active_start:active_stop], 0.12)
np.testing.assert_allclose(dc_current_nA[active_stop:], 0.0)

sine_block = results["Sine wave"].samples["current"]
sine_time_ms = np.asarray(sine_block.time.to_decimal(u.ms))
sine_current_nA = np.asarray(sine_block.values.to_decimal(u.nA))[:, 0]
np.testing.assert_allclose(sine_current_nA[:active_start], 0.0)
np.testing.assert_allclose(sine_current_nA[active_stop:], 0.0)

arbitrary_current_nA = np.asarray(
    results["Arbitrary biphasic"].samples["current"].values.to_decimal(u.nA)
)[:, 0]
assert arbitrary_current_nA.min() < 0.0 < arbitrary_current_nA.max()

In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(13.0, 6.0), sharex="col", constrained_layout=True)
colors = ("#2478b5", "#d1495b", "#2a9d6f")

for column, ((label, result), color) in enumerate(zip(results.items(), colors)):
    current = result.samples["current"]
    voltage = result.samples["voltage"]
    current_time_ms = np.asarray(current.time.to_decimal(u.ms))
    voltage_time_ms = np.asarray(voltage.time.to_decimal(u.ms))
    current_nA = np.asarray(current.values.to_decimal(u.nA))[:, 0]
    voltage_mV = np.asarray(voltage.values.to_decimal(u.mV))[:, 0]

    axes[0, column].step(current_time_ms, current_nA, where="mid", color=color)
    axes[0, column].axhline(0.0, color="0.75", linewidth=0.8)
    axes[0, column].set_title(label)
    axes[1, column].plot(voltage_time_ms, voltage_mV, color=color)
    axes[1, column].set_xlabel("Time (ms)")

axes[0, 0].set_ylabel("Injected current (nA)")
axes[1, 0].set_ylabel("Membrane voltage (mV)")
plt.show()